In [1]:
import datasets
import keras_hub
import transformers
import numpy as np
import tensorflow as tf
import tqdm.notebook as tqdm
import sklearn.model_selection
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds
import keras
from collections.abc import Callable

2025-09-29 18:58:15.522067: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-29 18:58:15.529307: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759136295.538362  561634 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759136295.541395  561634 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759136295.548261  561634 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except:
        pass

In [3]:
train = datasets.load_dataset('wangrongsheng/ag_news', split='train')
test  = datasets.load_dataset('wangrongsheng/ag_news', split='test')

In [4]:
tokenizer = transformers.AutoTokenizer.from_pretrained('gpt2')

In [5]:
tokenizer.pad_token = tokenizer.eos_token

In [6]:
def get_name(pref: str | None, suf: str, sep: str = '_') -> str | None:
    return pref and pref + sep + suf

In [24]:
tokens_train = []
for text in train['text']:
    tokens_train.append(tokenizer(text, max_length=max_size, padding="max_length", truncation=True)['input_ids'])

In [ ]:
tokens_test = []
for text in test['text']:
    tokens_test.append(tokenizer(text, max_length=max_size, padding="max_length", truncation=True)['input_ids'])

In [ ]:
tokens_train = np.array(tokens_train)
tokens_test = np.array(tokens_test)

In [ ]:
train_y = np.array(train["label"])
test_y = np.array(test["label"])

In [36]:
def get_model(
    n_tokens,
    vocab_size,
    embedding_dim,
    num_classes,
    name,
):
    inputs = keras.layers.Input((n_tokens,), dtype="int32", name=get_name(name, "inputs"))
    embed = keras.layers.Embedding(vocab_size, embedding_dim, name=get_name(name, "embedding"))
    x = embed(inputs)
    
    # x = keras.layers.LSTM(embedding_dim, name=get_name(name, "lstm"), return_state=False)(x)
    
    # x = keras.layers.RNN(keras.layers.LSTMCell(embedding_dim), name=get_name(name, "lstm"), return_state=False)(x)
    
    # x = keras.layers.RNN(keras.layers.StackedRNNCells([
    #     keras.layers.LSTMCell(embedding_dim),
    #     keras.layers.LSTMCell(embedding_dim),
    # ]), name=get_name(name, "wtf"), return_state=False)(x)

    # x = keras.layers.Bidirectional(keras.layers.RNN(keras.layers.StackedRNNCells([
    #     keras.layers.LSTMCell(embedding_dim),
    #     keras.layers.LSTMCell(embedding_dim),
    # ]), name=get_name(name, "wtf"), return_state=False))(x)

    # Перевод
    input2 = keras.layers.Input((n_tokens,), dtype="float32", name=get_name(name, "input2"))
    y = embed(input2)
    x, s = keras.layers.LSTM(embedding_dim, name=get_name(name, "lstm"), return_state=True)(x)
    x = keras.layers.LSTM(embedding_dim, name=get_name(name, "lstm"), return_state=False)(y, initial_state=s)
    
    x = keras.layers.Dense(4, name=get_name(name, "dense"))(x)
    return keras.Model(
        inputs=inputs,
        outputs=x,
        name=name,
    )

In [37]:
model = get_model(
    121,
    len(tokenizer.get_vocab()),
    128,
    4,
    "lstm",
)

In [38]:
max_size = 121

In [39]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [40]:
model.fit(tokens_train, train_y, batch_size=128, epochs=10)

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 33s 31ms/step - accuracy: 0.8768 - loss: 0.3467
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 28s 30ms/step - accuracy: 0.9429 - loss: 0.1716
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 28s 30ms/step - accuracy: 0.9594 - loss: 0.1204
Epoch 4/10
341/938 ━━━━━━━━━━━━━━━━━━━━ 17s 30ms/step - accuracy: 0.9742 - loss: 0.0765

KeyboardInterrupt: 